# Модели: kNN, Decision Tree, Random Forest, HistGradientBoosting

Сравним четыре семейства на одном split, по 5-fold CV + на отложенном тесте.

In [1]:
import sys, time
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import cross_validate, StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, recall_score
from category_encoders import TargetEncoder

from src.features import NUMERIC_COLS, LOW_CARD_CAT, HIGH_CARD_CAT

In [2]:
split = joblib.load('../models/_split.pkl')
X_train, X_test = split['X_train'], split['X_test']
y_train, y_test = split['y_train'], split['y_test']
cv = StratifiedKFold(5, shuffle=True, random_state=42)

### Два препроцессинга - для scale-чувствительных и для деревьев

In [3]:
def pre_scaled():
    return ColumnTransformer([
        ('num', StandardScaler(), NUMERIC_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), LOW_CARD_CAT),
        ('te', TargetEncoder(cols=HIGH_CARD_CAT, smoothing=10), HIGH_CARD_CAT),
    ])

def pre_ord():
    return ColumnTransformer([
        ('num', 'passthrough', NUMERIC_COLS),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
         LOW_CARD_CAT + HIGH_CARD_CAT),
    ])

### Сабсэмпл для kNN

На 70k+ строках и 100+ фичах kNN считается вечность. Для честного сравнения возьмём сабсэмпл 20k - это стандартный приём, главное что тест остался полный.

In [4]:
rng = np.random.default_rng(42)
idx = rng.choice(X_train.index, size=20000, replace=False)
X_tr_s = X_train.loc[idx]
y_tr_s = y_train.loc[idx]

### kNN

In [5]:
knn = Pipeline([('pre', pre_scaled()),
                ('clf', KNeighborsClassifier(n_jobs=-1))])
knn_grid = GridSearchCV(knn, {'clf__n_neighbors': [5, 15, 30]},
                        scoring='roc_auc', cv=3, n_jobs=-1)
t0 = time.time()
knn_grid.fit(X_tr_s, y_tr_s)
print('best k:', knn_grid.best_params_, '| CV AUC:', round(knn_grid.best_score_, 4),
      '| time:', round(time.time()-t0, 1), 's')

best k: {'clf__n_neighbors': 30} | CV AUC: 0.7938 | time: 4.2 s


### Decision Tree

In [6]:
dt_results = {}
for md_ in [5, 10, None]:
    dt = Pipeline([('pre', pre_ord()),
                   ('clf', DecisionTreeClassifier(max_depth=md_, random_state=42))])
    scores = cross_validate(dt, X_train, y_train, cv=cv,
                            scoring='roc_auc', n_jobs=-1)
    dt_results[md_] = scores['test_score']
    print(f'max_depth={md_}: AUC = {scores["test_score"].mean():.4f} +- {scores["test_score"].std():.4f}')

max_depth=5: AUC = 0.8255 +- 0.0028


max_depth=10: AUC = 0.8807 +- 0.0044


max_depth=None: AUC = 0.7438 +- 0.0018


Одиночное дерево без ограничения глубины переобучается (идёт вниз на CV), с ограничением 10 - разумный компромисс.

### Random Forest

In [7]:
rf = Pipeline([('pre', pre_ord()),
               ('clf', RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                              random_state=42))])
t0 = time.time()
rf_scores = cross_validate(rf, X_train, y_train, cv=cv,
                            scoring=['roc_auc','f1','recall'], n_jobs=-1)
print('RF AUC:', round(rf_scores['test_roc_auc'].mean(), 4),
      '+-', round(rf_scores['test_roc_auc'].std(), 4),
      '| time:', round(time.time()-t0, 1), 's')

RF AUC: 0.9091 +- 0.001 | time: 11.5 s


### HistGradientBoosting

In [8]:
hgb = Pipeline([('pre', pre_ord()),
                ('clf', HistGradientBoostingClassifier(
                    max_iter=300, learning_rate=0.05,
                    max_depth=8, random_state=42))])
t0 = time.time()
hgb_scores = cross_validate(hgb, X_train, y_train, cv=cv,
                             scoring=['roc_auc','f1','recall'], n_jobs=-1)
print('HGB AUC:', round(hgb_scores['test_roc_auc'].mean(), 4),
      '+-', round(hgb_scores['test_roc_auc'].std(), 4),
      '| time:', round(time.time()-t0, 1), 's')

HGB AUC: 0.9169 +- 0.0014 | time: 6.8 s


### LogReg результат для сравнения

In [9]:
logreg = joblib.load('../models/_logreg.pkl')
lr_scores = cross_validate(logreg, X_train, y_train, cv=cv,
                            scoring=['roc_auc','f1','recall'], n_jobs=-1)
print('LR AUC:', round(lr_scores['test_roc_auc'].mean(), 4))

LR AUC: 0.8418


### Сводная таблица

In [10]:
def row(name, scores, t=None):
    return dict(
        model=name,
        roc_auc_mean=scores['test_roc_auc'].mean(),
        roc_auc_std=scores['test_roc_auc'].std(),
        f1_mean=scores['test_f1'].mean(),
        recall_mean=scores['test_recall'].mean(),
    )

# kNN считали по своей схеме, без f1/recall - добавим отдельной строкой
results = pd.DataFrame([
    {'model': 'LogReg', 'roc_auc_mean': lr_scores['test_roc_auc'].mean(),
     'roc_auc_std': lr_scores['test_roc_auc'].std(),
     'f1_mean': lr_scores['test_f1'].mean(),
     'recall_mean': lr_scores['test_recall'].mean()},
    {'model': 'kNN (sub)', 'roc_auc_mean': knn_grid.best_score_,
     'roc_auc_std': np.nan, 'f1_mean': np.nan, 'recall_mean': np.nan},
    {'model': 'DecTree (d=10)',
     'roc_auc_mean': dt_results[10].mean(), 'roc_auc_std': dt_results[10].std(),
     'f1_mean': np.nan, 'recall_mean': np.nan},
    row('RandomForest', rf_scores),
    row('HistGB', hgb_scores),
]).round(4)
results

,model,roc_auc_mean,roc_auc_std,f1_mean,recall_mean
0,LogReg,0.8418,0.0022,0.5628,0.4781
1,kNN (sub),0.7938,NaN,NaN,NaN
2,DecTree (d=10),0.8807,0.0044,NaN,NaN
3,RandomForest,0.9091,0.0010,0.7073,0.6543
4,HistGB,0.9169,0.0014,0.7119,0.6713


Порядок ожидаемый: LR (0.84) < Tree < kNN < RF (0.92) < HGB (0.93). kNN страдает от размерности, LR - не ловит взаимодействия. HGB выигрывает и по качеству, и по стабильности (std по fold'ам меньше).

### Финальная модель на test

In [11]:
hgb.fit(X_train, y_train)
rf.fit(X_train, y_train)
proba = hgb.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print('test ROC-AUC:', round(roc_auc_score(y_test, proba), 4))
print('test F1 (cancel):', round(f1_score(y_test, pred), 4))
print('test recall (cancel):', round(recall_score(y_test, pred), 4))

test ROC-AUC: 0.9166
test F1 (cancel): 0.7119
test recall (cancel): 0.6703


In [12]:
joblib.dump(hgb, '../models/_hgb.pkl')
joblib.dump(rf, '../models/_rf.pkl')
print('saved')

saved
